# Bayesian Model Comparison

Compare local-evidence classifiers on the same parent-text partitions and cached windows.

In [ ]:
from __future__ import annotations

import matplotlib.pyplot as plt
from dataclasses import asdict
import numpy as np
import os
from pathlib import Path

from IPython.display import display
import pandas as pd
from tqdm.auto import tqdm

from local_evidence_comparison_utils import (
    expand_model_scores_to_tokens,
    plot_model_score_heatmap,
    plot_normalized_token_metrics_with_model_evidence,
    select_comparison_texts,
)
from local_evidence_models_utils import (
    BayesianConfig,
    GaussianMixtureNBConfig,
    LOCAL_FEATURE_COLUMNS,
    MODEL_NAMES,
    fit_local_evidence_classifier,
    run_local_evidence_model_comparison,
    score_local_evidence_classifier,
    training_input_fingerprint,
)
from local_evidence_utils import (
    default_local_evidence_path,
    load_or_build_local_evidence_windows,
    split_local_evidence_texts,
)
from semeval_boundary_utils import (
    build_gold_mg_regions,
    evaluate_mg_regions,
    predict_mg_regions,
    summarize_mg_region_evaluations,
)
from token_candidate_similarity_heatmaps_utils import (
    load_similarity_row,
    load_text_row,
    load_token_labels,
    plot_focused_token_metrics_with_local_evidence,
    plot_normalized_token_metrics_with_local_evidence,
    text_index_for_id,
    validated_text_summary,
    visualize_text_similarities,
)

DATA_ROOT = Path(os.environ.get("TEXT_DETECTION_DATA", "D:/Projects/master_diploma/text-detection-data"))
PROCESSED_ROOT = DATA_ROOT / "processed"
DATASET = "kaggle_train"
SEMEVAL_DATASET = "semeval2024_task8_subtask_c_dev"
SEMEVAL_SPLIT = "MG"
SEMEVAL_PREPARED_PATH = DATA_ROOT / "raw/semevals_2024-task8/SemEval2024-task8/subtaskC/SemEval2024_task8_C_dev_MG.csv"
MODEL = "gemma_3_1b_it"

WINDOW_LENGTHS = (32, 64, 128)
MODEL_NAMES = ("logistic_regression", "gaussian_nb", "gaussian_mixture_nb", "pymc_bayesian_logistic")
LOCAL_EVIDENCE_MODEL = "logistic_regression"
STRIDE = 16
TOP_K = 20
SMOOTHING_WINDOW = 10
SEMEVAL_SMOOTHING_WINDOW = 5
SEMEVAL_MINIMUM_REGION_LENGTH = 3
SEED = 42
SEMEVAL_MG_SCORE_THRESHOLD = 0.5
SEMEVAL_INSPECTION_MODEL = "logistic_regression"
SEMEVAL_INSPECTION_WINDOW_LENGTH = 64
SEMEVAL_SELECTED_TEXTS = ()
ADVI_ITERATIONS = 10_000
POSTERIOR_DRAWS = 1_000
SELECTED_TEXTS = ()
RECOMPUTE_WINDOWS = False
SAVE_FITTED_MODELS = False
REUSE_FITTED_MODELS = True
TRUST_FITTED_MODEL_CACHE = False
try:
    import pymc
except ModuleNotFoundError:
    PYMC_AVAILABLE = False
else:
    PYMC_AVAILABLE = True
if not PYMC_AVAILABLE:
    display("PyMC unavailable; continuing with LR, GaussianNB, and GaussianMixtureNB. Install it with: uv sync --extra bayesian")
AVAILABLE_MODELS = tuple(name for name in MODEL_NAMES if name != "pymc_bayesian_logistic" or PYMC_AVAILABLE)
BAYESIAN_CONFIG = BayesianConfig(
    advi_iterations=ADVI_ITERATIONS,
    posterior_draws=POSTERIOR_DRAWS,
    random_seed=SEED,
)
GAUSSIAN_MIXTURE_COMPONENTS = 2
GAUSSIAN_MIXTURE_CONFIG = GaussianMixtureNBConfig(
    components=GAUSSIAN_MIXTURE_COMPONENTS,
    reg_covar=1e-6,
)
MODEL_CACHE_ROOT = PROCESSED_ROOT / f"{DATASET}__{MODEL}" / "local_evidence" / "models"
MODEL_CACHE_MANIFEST = {
    "dataset": DATASET,
    "model": MODEL,
    "features": LOCAL_FEATURE_COLUMNS,
    "seed": SEED,
    "window_lengths": WINDOW_LENGTHS,
    "model_names": AVAILABLE_MODELS,
    "bayesian_config": asdict(BAYESIAN_CONFIG),
    "gaussian_mixture_config": asdict(GAUSSIAN_MIXTURE_CONFIG),
}
pd.set_option("display.max_columns", 100)

In [ ]:
WINDOWS_PATH = default_local_evidence_path(PROCESSED_ROOT, DATASET, MODEL)
windows = load_or_build_local_evidence_windows(
    PROCESSED_ROOT,
    DATASET,
    MODEL,
    path=WINDOWS_PATH,
    window_lengths=WINDOW_LENGTHS,
    stride=STRIDE,
    top_k=TOP_K,
    recompute=RECOMPUTE_WINDOWS,
)
display(windows.groupby(["split", "window_length"]).size().rename("window_rows").reset_index())

In [ ]:
partitions = split_local_evidence_texts(windows, seed=SEED)
MODEL_CACHE_MANIFEST["training_input_fingerprint"] = training_input_fingerprint(windows, partitions)
fitted_models = {}
comparison_summary, comparison_predictions, scored_windows, coefficient_summaries = run_local_evidence_model_comparison(
    windows,
    partitions,
    AVAILABLE_MODELS,
    feature_columns=LOCAL_FEATURE_COLUMNS,
    seed=SEED,
    bayesian_config=BAYESIAN_CONFIG,
    gaussian_mixture_config=GAUSSIAN_MIXTURE_CONFIG,
    fitted_models=fitted_models,
    cache_root=MODEL_CACHE_ROOT,
    cache_manifest=MODEL_CACHE_MANIFEST,
    reuse_fitted_models=REUSE_FITTED_MODELS,
    save_fitted_models=SAVE_FITTED_MODELS,
    trust_fitted_models=TRUST_FITTED_MODEL_CACHE,
)
display(
    comparison_summary.sort_values(["model", "window_length", "aggregation"])[
        ["model", "window_length", "aggregation", "accuracy", "macro_f1"]
    ].reset_index(drop=True)
)
display(coefficient_summaries)
display(comparison_summary.groupby(["model", "window_length", "aggregation"]).size())

In [ ]:
SEMEVAL_WINDOWS_PATH = default_local_evidence_path(PROCESSED_ROOT, SEMEVAL_DATASET, MODEL)
semeval_windows = load_or_build_local_evidence_windows(
    PROCESSED_ROOT,
    SEMEVAL_DATASET,
    MODEL,
    path=SEMEVAL_WINDOWS_PATH,
    window_lengths=WINDOW_LENGTHS,
    stride=STRIDE,
    top_k=TOP_K,
    splits=(SEMEVAL_SPLIT,),
    recompute=RECOMPUTE_WINDOWS,
)
if semeval_windows.empty:
    raise ValueError("No SemEval windows are available; process the SemEval MG split through stages 1 and 2 first.")
semeval_annotations = pd.read_csv(SEMEVAL_PREPARED_PATH)
if {"id", "text", "boundary_word_index"}.difference(semeval_annotations.columns):
    raise ValueError("Prepared SemEval CSV must contain id, text, and boundary_word_index columns.")
display(semeval_windows.groupby("window_length").size().rename("window_rows").reset_index())

In [ ]:
semeval_scored_frames = []
for model_name in tqdm(AVAILABLE_MODELS, desc="Scoring SemEval with trained models", unit="model"):
    fitted_model_lengths = [
        window_length for candidate_model, window_length in fitted_models if candidate_model == model_name
    ]
    for window_length in tqdm(fitted_model_lengths, desc=f"{model_name} window lengths", leave=False, unit="window"):
        evaluation_windows = semeval_windows.loc[semeval_windows["window_length"].eq(window_length)]
        if evaluation_windows.empty:
            continue
        semeval_scored_frames.append(
            score_local_evidence_classifier(
                fitted_models[(model_name, window_length)], evaluation_windows, LOCAL_FEATURE_COLUMNS
            )
        )
if not semeval_scored_frames:
    raise ValueError("No matching Kaggle and SemEval window lengths are available for transfer validation.")
semeval_scored_windows = pd.concat(semeval_scored_frames, ignore_index=True)

In [ ]:
semeval_token_labels = load_token_labels(
    PROCESSED_ROOT, SEMEVAL_DATASET, MODEL, SEMEVAL_SPLIT, show_progress=True
)
semeval_token_counts = {text_id: len(labels) for text_id, labels in semeval_token_labels.items()}
available_semeval_annotations = semeval_annotations.loc[semeval_annotations["id"].isin(semeval_token_labels)].copy()
display(f"Using {len(available_semeval_annotations)} of {len(semeval_annotations)} SemEval annotations with processed tokens")
semeval_predicted_regions = predict_mg_regions(
    semeval_scored_windows,
    semeval_token_counts,
    threshold=SEMEVAL_MG_SCORE_THRESHOLD,
    smoothing_window=SEMEVAL_SMOOTHING_WINDOW,
    minimum_region_length=SEMEVAL_MINIMUM_REGION_LENGTH,
    show_progress=True,
)
semeval_gold_regions = build_gold_mg_regions(available_semeval_annotations, semeval_token_labels)
semeval_region_evaluations = evaluate_mg_regions(semeval_predicted_regions, semeval_gold_regions)
semeval_region_summary = summarize_mg_region_evaluations(semeval_region_evaluations)
display(semeval_region_summary.sort_values(["mg_f1", "model", "window_length"]))
display(semeval_region_evaluations[["model", "window_length", "text_id", "mg_f1", "predicted_mg_region_count", "spurious_mg_region_count", "missed_mg_region_count", "boundary_error"]].sort_values(["mg_f1", "spurious_mg_region_count", "text_id"]))

In [ ]:
semeval_inspection = semeval_region_evaluations.loc[
    semeval_region_evaluations["model"].eq(SEMEVAL_INSPECTION_MODEL)
    & semeval_region_evaluations["window_length"].eq(SEMEVAL_INSPECTION_WINDOW_LENGTH)
].copy()
if SEMEVAL_SELECTED_TEXTS:
    semeval_inspection = semeval_inspection.loc[semeval_inspection["text_id"].isin(SEMEVAL_SELECTED_TEXTS)]
else:
    semeval_inspection = semeval_inspection.nsmallest(3, "mg_f1")
for prediction in semeval_inspection.itertuples(index=False):
    score_timeline = semeval_scored_windows.loc[
        semeval_scored_windows["model"].eq(prediction.model)
        & semeval_scored_windows["window_length"].eq(prediction.window_length)
        & semeval_scored_windows["text_id"].eq(prediction.text_id)
    ].sort_values("start_token")
    token_scores = expand_model_scores_to_tokens(score_timeline, prediction.token_count)
    token_scores["smoothed_mg_score"] = (token_scores["mg_score"].rolling(SEMEVAL_SMOOTHING_WINDOW, center=True, min_periods=1).median())
    token_labels = semeval_token_labels[prediction.text_id]
    figure, (axis, gold_zone_axis, predicted_zone_axis, text_axis) = plt.subplots(4, 1, figsize=(14, 7), sharex=True, constrained_layout=True, gridspec_kw={"height_ratios": [4, 1, 1, 1]})
    axis.plot(token_scores["token_position"], token_scores["mg_score"], color="tab:orange", alpha=0.8, linewidth=1, marker=".", markersize=3, zorder=3, label="raw token MG score")
    axis.plot(token_scores["token_position"], token_scores["smoothed_mg_score"], zorder=2, label="smoothed token MG score")
    axis.axhline(SEMEVAL_MG_SCORE_THRESHOLD, color="tab:gray", linestyle="--", label="MG threshold")
    gold_zone_axis.set_ylabel("Gold")
    predicted_zone_axis.set_ylabel("Predicted")
    text_axis.set_ylabel("Text")
    for region in semeval_gold_regions.loc[semeval_gold_regions["text_id"].eq(prediction.text_id)].itertuples(index=False):
        color = "tab:green" if region.label == "MG" else "tab:gray"
        gold_zone_axis.axvspan(region.start_token, region.stop_token, color=color, alpha=0.75)
        gold_zone_axis.text((region.start_token + region.stop_token) / 2, 0.5, region.label, ha="center", va="center", color="white", fontweight="bold")
    for region in semeval_predicted_regions.loc[
        semeval_predicted_regions["model"].eq(prediction.model)
        & semeval_predicted_regions["window_length"].eq(prediction.window_length)
        & semeval_predicted_regions["text_id"].eq(prediction.text_id)
    ].itertuples(index=False):
        color = "tab:red" if region.label == "MG" else "tab:blue"
        predicted_zone_axis.axvspan(region.start_token, region.stop_token, color=color, alpha=0.75)
        predicted_zone_axis.text((region.start_token + region.stop_token) / 2, 0.5, region.label, ha="center", va="center", color="white", fontweight="bold")
        if region.start_token > 0:
            direction = "HG-to-MG" if region.label == "MG" else "MG-to-HG"
            axis.axvline(region.start_token, color=color, linestyle="--", alpha=0.8, label=direction)
            predicted_zone_axis.axvline(region.start_token, color=color, linestyle="--", alpha=0.8)
    for start_token in range(0, len(token_labels), 12):
        stop_token = min(start_token + 12, len(token_labels))
        text_axis.text(start_token, 0.5, "".join(token_labels[start_token:stop_token]), ha="left", va="center", fontsize=8)
    for zone_axis in (gold_zone_axis, predicted_zone_axis, text_axis):
        zone_axis.set_ylim(0, 1)
        zone_axis.set_yticks([])
        zone_axis.grid(False)
    axis.set(
        title=f"SemEval regions: text {prediction.text_id}, MG F1={prediction.mg_f1:.3f}",
        xlabel="Token position",
        ylabel="MG score",
        ylim=(0, 1),
    )
    axis.legend()
    display(figure)

In [ ]:
test_predictions = comparison_predictions.query('partition == "test"').copy()
if SELECTED_TEXTS:
    selected_predictions = test_predictions.loc[test_predictions["text_id"].isin(SELECTED_TEXTS)]
else:
    selected_predictions = select_comparison_texts(test_predictions, count=3, seed=SEED)
display(selected_predictions)
display(
    selected_predictions.assign(
        is_error=lambda frame: frame["prediction"].ne(frame["label"])
    ).loc[:, ["model", "window_length", "aggregation", "text_id", "label", "prediction", "is_error"]]
)

In [ ]:
for _, selected in selected_predictions.drop_duplicates(["split", "text_id"]).iterrows():
    text_id = int(selected["text_id"])
    split = selected["split"]
    text_index = text_index_for_id(PROCESSED_ROOT, DATASET, MODEL, split, text_id)
    text_row = load_text_row(PROCESSED_ROOT, DATASET, MODEL, split, text_index)
    similarity_row = load_similarity_row(PROCESSED_ROOT, DATASET, MODEL, split, text_id)
    token_summary, token_labels = validated_text_summary(text_row, similarity_row, top_k=TOP_K)
    for window_length in WINDOW_LENGTHS:
        text_windows = scored_windows.loc[
            scored_windows["partition"].eq("test")
            & scored_windows["split"].eq(split)
            & scored_windows["text_id"].eq(text_id)
            & scored_windows["window_length"].eq(window_length)
        ]
        if text_windows.empty:
            continue
        local_evidence_windows = text_windows.loc[
            text_windows["model"].eq(LOCAL_EVIDENCE_MODEL)
        ]
        if local_evidence_windows.empty:
            raise ValueError(f"selected local evidence model is unavailable: {LOCAL_EVIDENCE_MODEL}")
        display(plot_model_score_heatmap(text_windows, len(token_labels), f"Model scores: text {text_id}, window {window_length}"))
        display(plot_normalized_token_metrics_with_model_evidence(
            token_summary, text_windows, len(token_labels), window=SMOOTHING_WINDOW
        ))
        display(f"Local evidence model: {LOCAL_EVIDENCE_MODEL}")
        display(plot_normalized_token_metrics_with_local_evidence(
            token_summary, local_evidence_windows, len(token_labels), window=SMOOTHING_WINDOW
        ))
        display(plot_focused_token_metrics_with_local_evidence(
            token_summary,
            local_evidence_windows,
            len(token_labels),
            window=SMOOTHING_WINDOW,
        ))
        visualize_text_similarities(
            PROCESSED_ROOT, DATASET, MODEL, split, text_index, top_k=TOP_K,
            token_range=(int(text_windows["start_token"].min()), int(text_windows["stop_token"].max())),
        )
        finite_intervals = text_windows.loc[
            np.isfinite(text_windows["mg_score_lower"])
            & np.isfinite(text_windows["mg_score_upper"])
        ]
        if not finite_intervals.empty:
            display(f"PyMC posterior predictive bands available for text {text_id}")